## Flight radard 
## 31.8.2026

- OpenSky network API
    - https://openskynetwork.github.io/opensky-api/rest.html
    - 400 kreditů/den anonymně → 4 000 po registraci → 8 000 active feeder (≥30 % uptime)
    - Feeder status se přepočítává každých ~2 h; tier upgrade po ~50 requestech
    - Feeder kvótu potvrdí `X-Rate-Limit-Remaining` > 4 000 na začátku dne
- ADS-B
    - receiver (pouze anténa + RPi, pasivní příjem)
    - Exchange (sharování dat přes internet s ostatními)


### TODO:
- do GITHUBU, do DASHBOARDU
- check toho obnovování tokenů, rate, a python API obecně (hotovo: OAuth2 TokenManager + 401 retry, oficiální `OpenSkyApi` s credentials)
- přidat varování na nové letadlo (ze 157 jich je 104 nových)
- přidat varování na nestandardní model/kateogrii/rychlost
- přidat varování že nový rekord (rychlost, výška)
- z callsign jde doplnit spolešnost, EJ = easyjet, atd
- doplnit typ letadla jde z https://www.planespotters.net nebo https://globe.adsbexchange.com

In [ ]:
import os
os.chdir("c:\\Users\\jirka\\Documents\\MyProjects\\flight_radar")

import pandas as pd

from IPython.display import display

from flight_db import OpenSkyClient, fetch_aircraft, fetch_observations


In [5]:

#? PRAHA
# params={"lamin": 49.8, "lamax": 50.5, "lomin": 13.8, "lomax": 15.0}
#? ČR
params={"lamin": 48.4, "lamax": 51.2, "lomin": 11.8, "lomax": 19.0 }

client = OpenSkyClient(params=params,save_to_db=True)
data = client.fetch()
client.print_states()

OpenSky time=2026-09-02 16:19:20.000000  letadel v odpovědi=159  nových pozorování=159  přeskočeno (stejný snapshot)=0  uloženo do DB=True
kredity remaining=3974  feeder_confirmed=False
SPFLT      489572 lat=50.2632 lon=18.732 baro=457.2 m geo=None m spd=44.92 m/s trk=283.24° vrt=-0.33 m/s on_ground=False squawk=7000 country=Poland category=0 last_contact=2026-09-02 16:19:19.000000 time_pos=2026-09-02 16:19:19.000000 spi=False pos=0 sensors=None
PGT972     4bc8d6 lat=48.5002 lon=12.0762 baro=10553.7 m geo=10911.84 m spd=262.2 m/s trk=102.81° vrt=4.55 m/s on_ground=False squawk=7651 country=Turkey category=0 last_contact=2026-09-02 16:19:19.000000 time_pos=2026-09-02 16:19:19.000000 spi=False pos=0 sensors=None
AUA235D    44029f lat=50.232 lon=15.1329 baro=9144 m geo=9418.32 m spd=213.04 m/s trk=338.76° vrt=0 m/s on_ground=False squawk=1000 country=Austria category=0 last_contact=2026-09-02 16:19:19.000000 time_pos=2026-09-02 16:19:19.000000 spi=False pos=0 sensors=None
PGT839Q    4bc8c

In [ ]:

#? start 1.9. "večer"
# remaining > 4000 na začátku dne = feeder kvóta 8000.
# Status feederu se přepočítává každých ~2 h; upgrade se projeví po ~50 requestech.
#? 2.9. 12:30 mám 3.979.   v 18:19 stále 3973
quota = client.check_quota()

X-Rate-Limit-Remaining=3973 <= 4000 - standardní kvóta, nebo feeder tier ještě nenasadil (přepočet každých ~2 h, upgrade po ~50 requestech).
remaining=3973  feeder_confirmed=False  HTTP 200  retry_after=None
  X-Rate-Limit-Remaining: 3973


In [7]:

def _rows_to_frame(rows):
    records = [dict(row) for row in rows]
    if pd is None:
        return records
    return pd.DataFrame.from_records(records)


print("aircraft")
display(_rows_to_frame(fetch_aircraft()))

print("observations (posledních 50)")
display(_rows_to_frame(fetch_observations(limit=50)))

aircraft


,icao24,callsign,origin_country,category,first_seen,last_seen,observation_count,note
0,49f0e1,TXLU04,Czech Republic,0,2026-08-31 21:38:09.000000,2026-09-02 16:19:20.000000,22,NaN
1,440821,AUA277A,Austria,0,2026-09-01 15:03:16.000000,2026-09-02 16:19:20.000000,3,NaN
2,49d5a6,TVS2CD,Czech Republic,0,2026-09-01 15:03:16.000000,2026-09-02 16:19:20.000000,10,NaN
3,48ad08,LOT7TB,Poland,0,2026-09-01 15:03:16.000000,2026-09-02 16:19:20.000000,6,NaN
4,440051,EJU97JM,Austria,0,2026-09-01 15:14:04.000000,2026-09-02 16:19:20.000000,3,NaN
...,...,...,...,...,...,...,...,...
876,3e4ebc,DIVAA,Germany,0,2026-09-01 15:14:04.000000,2026-09-01 15:14:04.000000,1,NaN
877,49d590,AAO57,Czech Republic,0,2026-09-01 15:03:16.000000,2026-09-01 15:03:16.000000,2,NaN
878,49d0c6,ABP711,Czech Republic,0,2026-09-01 15:03:16.000000,2026-09-01 15:03:16.000000,2,NaN
879,70612f,JZR38,Kuwait,0,2026-09-01 15:03:16.000000,2026-09-01 15:03:16.000000,2,NaN


observations (posledních 50)


,id,icao24,snapshot_time,ingested_at,callsign,origin_country,time_position,last_contact,longitude,latitude,...,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source,category
0,1406,4d200d,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,WZZ17ZC,Malta,2026-09-02 16:19:19.000000,2026-09-02 16:19:19.000000,16.3291,50.1890,...,0,214.58,268.63,5.53,None,10233.66,5450,0,0,0
1,1405,4692cc,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,AEE542,Greece,2026-09-02 16:19:20.000000,2026-09-02 16:19:20.000000,12.1971,48.9670,...,0,201.69,317.27,-0.33,None,11948.16,1000,0,0,0
2,1404,39856f,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,KLM43R,France,2026-09-02 16:19:19.000000,2026-09-02 16:19:20.000000,14.9115,49.3875,...,0,264.63,112.28,0.00,None,11544.30,1000,0,0,0
3,1403,a84ab1,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,UPS15,United States,2026-09-02 16:19:19.000000,2026-09-02 16:19:20.000000,15.4635,48.7915,...,0,229.92,295.02,0.00,None,10668.00,3212,0,0,0
4,1402,4d2067,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,WZZ44,Malta,2026-09-02 16:19:19.000000,2026-09-02 16:19:19.000000,13.0573,49.7944,...,0,197.49,302.99,-0.33,None,10675.62,4253,0,0,0
5,1401,4d2153,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,CXI2ZD,Malta,2026-09-02 16:19:19.000000,2026-09-02 16:19:19.000000,12.6844,48.4346,...,0,218.18,310.60,-7.48,None,8915.40,6340,0,0,4
6,1400,451d85,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,ASL19H,Bulgaria,2026-09-02 16:19:19.000000,2026-09-02 16:19:19.000000,18.7172,48.7602,...,0,218.57,339.90,0.00,None,11285.22,1167,0,0,0
7,1399,4691c1,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,AEE866,Greece,2026-09-02 16:19:19.000000,2026-09-02 16:19:19.000000,14.4985,50.1832,...,0,98.05,245.18,0.33,None,1303.02,4531,0,0,0
8,1398,4b19ff,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,SWR5TQ,Switzerland,2026-09-02 16:19:19.000000,2026-09-02 16:19:19.000000,15.2180,50.3616,...,0,233.78,64.45,-7.80,None,7292.34,1000,0,0,0
9,1397,4b19fc,2026-09-02 16:19:20.000000,2026-09-02 16:19:22.803595,SWR3WZ,Switzerland,2026-09-02 16:19:20.000000,2026-09-02 16:19:20.000000,16.7534,50.1487,...,0,249.82,53.63,-0.33,None,10911.84,1000,0,0,0


## Práce s PYTHON API

In [ ]:
# Oficiální klient NENÍ na PyPI (`opensky-api` 404). Instalace z GitHubu:
# %pip install "git+https://github.com/openskynetwork/opensky-api.git#subdirectory=python"
#
# Od 18. 3. 2026 OpenSky vyžaduje OAuth2 client_credentials.
# OpenSkyApi() bez credentials = anonymní přístup (snížená kvóta).
# Token si oficiální klient obnovuje sám (TokenManager, 30 s před expirací).

In [ ]:
from opensky_api import OpenSkyApi

from flight_db import get_tokens

# Stejné CLIENT_ID / CLIENT_SECRET jako REST klient (.env).
tm = get_tokens()
api = OpenSkyApi(client_id=tm.client_id, client_secret=tm.client_secret)

# bbox = (min lat, max lat, min lon, max lon) — Praha
states = api.get_states(bbox=(49.8, 50.5, 13.8, 15.0))

if states and states.states:
    print("Počet letadel:", len(states.states))
    for aircraft in states.states:
        print(
            aircraft.callsign,
            aircraft.icao24,
            aircraft.latitude,
            aircraft.longitude,
            aircraft.geo_altitude,
            aircraft.velocity,
            aircraft.true_track,
        )
else:
    print("Žádná data (None = chyba/rate-limit na straně oficiálního klienta).")